In [2]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================
# CONFIGURACIÓN
# ============================================================
ROOT_RESULTS = "Results_horas"
COL_REAL     = "Realidad"
COL_PRED     = "Prediccion"
OUTPUT_CSV   = "Results_horas/comparativa_arq_1h.csv"
TRIGGER      = 1000   # None = todos los instantes

FUENTES_BASELINE = [
    ("ARIMA", "results_baselines_1h", "test_results_", "_baselines.csv"),
    ("Naive", "results_baselines_1h", "test_results_", "_baselines.csv"),
]
# ============================================================

# Patrón nuevo: results_Arq{ID}_1h_{MODO}_W{W}
# Ejemplo:      results_Arq1_1h_A_W24
PATRON_ARQ = re.compile(r"^results_Arq(\d+)_1h_([AB])_W(\d+)$")


def mape(y_real, y_pred):
    mask = y_real > 1e-6
    return np.mean(np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])) * 100 if mask.sum() > 0 else np.nan


def calcular_metricas(y_real, y_pred, trigger=None):
    mask_nan = ~(np.isnan(y_real) | np.isnan(y_pred))
    y_real, y_pred = y_real[mask_nan], y_pred[mask_nan]
    if trigger is not None:
        m = y_real > trigger
        y_real, y_pred = y_real[m], y_pred[m]
    if len(y_real) == 0:
        return None, None, None, 0
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mp   = mape(y_real, y_pred)
    return round(mae, 2), round(rmse, 2), round(mp, 2), len(y_real)


def es_enlace_valido(enlace):
    partes = enlace.split("_")
    return len(partes) == 2 and all(len(p) >= 2 for p in partes)


def leer_carpeta_arq(carpeta_abs, prefijo):
    resultados = {}
    if not os.path.exists(carpeta_abs):
        return resultados
    for f in sorted(os.listdir(carpeta_abs)):
        if not f.startswith(prefijo) or not f.endswith(".csv"):
            continue
        enlace = f[len(prefijo):-4]
        if not enlace or not es_enlace_valido(enlace):
            continue
        try:
            df = pd.read_csv(os.path.join(carpeta_abs, f))
            if COL_REAL not in df.columns or COL_PRED not in df.columns:
                continue
            y_real = df[COL_REAL].values.astype(float)
            y_pred = df[COL_PRED].values.astype(float)
            mae, rmse, mp, n = calcular_metricas(y_real, y_pred, TRIGGER)
            if mae is not None:
                resultados[enlace] = {"MAE": mae, "RMSE": rmse, "MAPE": mp, "N": n}
        except Exception as e:
            print(f"   ❌ {f}: {e}")
    return resultados


def leer_baseline(etiqueta, carpeta_rel, prefijo, sufijo):
    col = f"{etiqueta}_t+1"
    carpeta_abs = os.path.join(ROOT_RESULTS, carpeta_rel)
    resultados = {}
    if not os.path.exists(carpeta_abs):
        return resultados
    for f in sorted(os.listdir(carpeta_abs)):
        if not f.startswith(prefijo) or not f.endswith(sufijo):
            continue
        enlace = f[len(prefijo): len(f) - len(sufijo)]
        if not enlace or not es_enlace_valido(enlace):
            continue
        try:
            df = pd.read_csv(os.path.join(carpeta_abs, f))
            if COL_REAL not in df.columns or col not in df.columns:
                continue
            y_real = df[COL_REAL].values.astype(float)
            y_pred = df[col].values.astype(float)
            mae, rmse, mp, n = calcular_metricas(y_real, y_pred, TRIGGER)
            if mae is not None:
                resultados[enlace] = {"MAE": mae, "RMSE": rmse, "MAPE": mp, "N": n}
        except Exception as e:
            print(f"   ❌ {f}: {e}")
    return resultados


def fila_global(df_enlaces, cols_modelo, metrica):
    fila = {"Enlace": "── GLOBAL ──"}
    for m in cols_modelo:
        col = f"{metrica}_{m}"
        fila[col] = round(df_enlaces[col].dropna().mean(), 2) if col in df_enlaces else None
    vals = {m: fila[f"{metrica}_{m}"] for m in cols_modelo if fila.get(f"{metrica}_{m}") is not None}
    fila[f"mejor_{metrica}"] = min(vals, key=vals.get) if vals else "-"
    return fila


def ordenar_columnas_modelo(cols_modelo):
    """
    Orden: ARIMA, Naive → Arq1_A → Arq1_B → Arq2_A → Arq2_B → ...
    """
    baselines_orden = ["ARIMA", "Naive"]
    baselines = [c for c in baselines_orden if c in cols_modelo]
    arq_cols  = [c for c in cols_modelo if c not in baselines_orden]

    def sort_key(etiq):
        # etiq tipo: Arq1_A  o  Arq2_B
        m = re.match(r"Arq(\d+)_([AB])", etiq)
        if m:
            modo_num = 0 if m.group(2) == "A" else 1
            return (int(m.group(1)), modo_num)
        return (99, 99)

    return baselines + sorted(arq_cols, key=sort_key)


def imprimir_tabla(df, metrica, cols_modelo, ventana, trigger):
    ancho = 14
    trigger_str = f" | Realidad > {trigger:,.0f}" if trigger else ""
    print(f"\n{'='*120}")
    print(f" {metrica} — horizonte t+1h | ventana W={ventana}{trigger_str}")
    print(f"{'='*120}")
    header = f"{'Enlace':>15}" + "".join(f"  {m:>{ancho}}" for m in cols_modelo) + f"  {'mejor':>{ancho}}"
    print(header)
    print("-" * len(header))

    for _, row in df[df["Enlace"] != "── GLOBAL ──"].iterrows():
        line = f"{row['Enlace']:>15}"
        for m in cols_modelo:
            val = row.get(f"{metrica}_{m}")
            line += f"  {f'{val:.2e}' if pd.notna(val) else '-':>{ancho}}"
        line += f"  {row.get(f'mejor_{metrica}', '-'):>{ancho}}"
        print(line)

    print("-" * len(header))
    row_g = df[df["Enlace"] == "── GLOBAL ──"].iloc[0]
    line  = f"{'── GLOBAL ──':>15}"
    for m in cols_modelo:
        val = row_g.get(f"{metrica}_{m}")
        line += f"  {f'{val:.2e}' if pd.notna(val) else '-':>{ancho}}"
    line += f"  {row_g.get(f'mejor_{metrica}', '-'):>{ancho}}"
    print(line)

    df_s = df[df["Enlace"] != "── GLOBAL ──"]
    col_mejor = f"mejor_{metrica}"
    if col_mejor in df_s.columns:
        print(f"\n🏆 VICTORIAS ({metrica}):")
        for modelo, n in df_s[col_mejor].value_counts().items():
            print(f"   {modelo}: {n}/{len(df_s)} ({100*n/len(df_s):.1f}%)")


def construir_tabla_enlaces(datos, cols_modelo, metricas=("MAE", "RMSE", "MAPE")):
    todos_enlaces = sorted(set(e for res in datos.values() for e in res))
    filas = []
    for enlace in todos_enlaces:
        fila = {"Enlace": enlace}
        for m in cols_modelo:
            if enlace in datos.get(m, {}):
                for met in metricas:
                    fila[f"{met}_{m}"] = datos[m][enlace][met]
                fila[f"N_{m}"] = datos[m][enlace]["N"]
            else:
                for met in metricas:
                    fila[f"{met}_{m}"] = None
                fila[f"N_{m}"] = 0
        filas.append(fila)

    df = pd.DataFrame(filas)

    for met in metricas:
        cols_met = [f"{met}_{m}" for m in cols_modelo if f"{met}_{m}" in df.columns]
        if cols_met:
            df[f"mejor_{met}"] = df[cols_met].idxmin(axis=1).str.replace(f"{met}_", "", regex=False)

    fila_g = {"Enlace": "── GLOBAL ──"}
    for met in metricas:
        g = fila_global(df, cols_modelo, met)
        for k, v in g.items():
            if k != "Enlace":
                fila_g[k] = v

    df = pd.concat([df, pd.DataFrame([fila_g])], ignore_index=True)
    return df


def comparar():
    # ── 1. Descubrir carpetas
    # Agrupamos por W → una tabla por ventana con todas las Arq y modos juntos
    # etiqueta: Arq{ID}_{MODO}  →  columnas: ARIMA, Naive, Arq1_A, Arq1_B, Arq2_A ...
    grupos = {}   # ventana → {etiqueta: datos}

    for nombre in sorted(os.listdir(ROOT_RESULTS)):
        m = PATRON_ARQ.match(nombre)
        if not m:
            continue
        arq_id, modo, w = m.group(1), m.group(2), m.group(3)
        ventana  = int(w)
        etiqueta = f"Arq{arq_id}_{modo}"   # sin "GRU_"

        if ventana not in grupos:
            grupos[ventana] = {}

        carpeta_abs     = os.path.join(ROOT_RESULTS, nombre)
        prefijo_fichero = f"test_Arq{arq_id}_1h_{modo}_W{w}_"
        print(f"📂 Leyendo {nombre}  →  etiqueta={etiqueta}")
        grupos[ventana][etiqueta] = leer_carpeta_arq(carpeta_abs, prefijo_fichero)
        print(f"   {len(grupos[ventana][etiqueta])} enlaces")

    if not grupos:
        print("❌ No se encontraron carpetas results_Arq*_1h_[AB]_W*")
        return

    # ── 2. Una tabla por ventana W ────────────────────────────
    todos_dfs = []

    for ventana, datos_arq in sorted(grupos.items()):
        print(f"\n{'#'*60}")
        print(f"  W={ventana} | H=1h  —  {len(datos_arq)} configuraciones")
        print(f"{'#'*60}")

        datos_all = dict(datos_arq)
        for etiq, carp_rel, pref, suf in FUENTES_BASELINE:
            datos_all[etiq] = leer_baseline(etiq, carp_rel, pref, suf)

        cols_baselines = [e for e, *_ in FUENTES_BASELINE]
        cols_modelo    = ordenar_columnas_modelo(cols_baselines + list(datos_arq.keys()))

        df_tabla = construir_tabla_enlaces(datos_all, cols_modelo)

        for metrica in ["MAE", "RMSE", "MAPE"]:
            imprimir_tabla(df_tabla, metrica, cols_modelo, ventana, TRIGGER)

        nombre_out = f"{ROOT_RESULTS}/comparativa_1h_W{ventana}.csv"
        df_tabla[df_tabla["Enlace"] != "── GLOBAL ──"].to_csv(nombre_out, index=False)
        print(f"\n💾 {nombre_out}")
        todos_dfs.append(df_tabla.assign(W=ventana))

    if todos_dfs:
        pd.concat(todos_dfs, ignore_index=True).to_csv(OUTPUT_CSV, index=False)
        print(f"\n💾 CSV global: {OUTPUT_CSV}")


if __name__ == "__main__":
    comparar()

📂 Leyendo results_Arq1_1h_A_W168  →  etiqueta=Arq1_A
   81 enlaces
📂 Leyendo results_Arq1_1h_A_W24  →  etiqueta=Arq1_A
   81 enlaces
📂 Leyendo results_Arq1_1h_B_W168  →  etiqueta=Arq1_B
   81 enlaces
📂 Leyendo results_Arq1_1h_B_W24  →  etiqueta=Arq1_B
   81 enlaces
📂 Leyendo results_Arq2_1h_A_W168  →  etiqueta=Arq2_A
   81 enlaces
📂 Leyendo results_Arq2_1h_A_W24  →  etiqueta=Arq2_A
   81 enlaces
📂 Leyendo results_Arq2_1h_B_W168  →  etiqueta=Arq2_B
   81 enlaces
📂 Leyendo results_Arq2_1h_B_W24  →  etiqueta=Arq2_B
   81 enlaces
📂 Leyendo results_Arq3_1h_A_W168  →  etiqueta=Arq3_A
   81 enlaces
📂 Leyendo results_Arq3_1h_A_W24  →  etiqueta=Arq3_A
   81 enlaces
📂 Leyendo results_Arq3_1h_B_W168  →  etiqueta=Arq3_B
   81 enlaces
📂 Leyendo results_Arq3_1h_B_W24  →  etiqueta=Arq3_B
   81 enlaces
📂 Leyendo results_Arq4_1h_A_W168  →  etiqueta=Arq4_A
   81 enlaces
📂 Leyendo results_Arq4_1h_A_W24  →  etiqueta=Arq4_A
   81 enlaces
📂 Leyendo results_Arq4_1h_B_W168  →  etiqueta=Arq4_B
   81 enlaces
📂 

In [1]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ============================================================
# CONFIGURACIÓN
# ============================================================
ROOT_RESULTS = "Results_horas"
COL_REAL     = "Realidad"
COL_PRED     = "Prediccion"
TRIGGER      = 1000   # None = todos los instantes

FUENTES_BASELINE = [
    ("ARIMA", "results_baselines_1h", "test_results_", "_baselines.csv"),
    ("Naive", "results_baselines_1h", "test_results_", "_baselines.csv"),
]
# ============================================================

PATRON_ARQ = re.compile(r"^results_Arq(\d+)_1h_([AB])_W(\d+)$")


def mape(y_real, y_pred):
    mask = y_real > 1e-6
    return np.mean(np.abs((y_real[mask] - y_pred[mask]) / y_real[mask])) * 100 if mask.sum() > 0 else np.nan


def calcular_metricas(y_real, y_pred, trigger=None):
    mask_nan = ~(np.isnan(y_real) | np.isnan(y_pred))
    y_real, y_pred = y_real[mask_nan], y_pred[mask_nan]
    if trigger is not None:
        m = y_real > trigger
        y_real, y_pred = y_real[m], y_pred[m]
    if len(y_real) == 0:
        return None, None, None, 0
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mp   = mape(y_real, y_pred)
    return round(mae, 2), round(rmse, 2), round(mp, 2), len(y_real)


def es_enlace_valido(enlace):
    partes = enlace.split("_")
    return len(partes) == 2 and all(len(p) >= 2 for p in partes)


def leer_carpeta_arq(carpeta_abs, prefijo):
    resultados = {}
    if not os.path.exists(carpeta_abs):
        return resultados
    for f in sorted(os.listdir(carpeta_abs)):
        if not f.startswith(prefijo) or not f.endswith(".csv"):
            continue
        enlace = f[len(prefijo):-4]
        if not enlace or not es_enlace_valido(enlace):
            continue
        try:
            df = pd.read_csv(os.path.join(carpeta_abs, f))
            if COL_REAL not in df.columns or COL_PRED not in df.columns:
                continue
            y_real = df[COL_REAL].values.astype(float)
            y_pred = df[COL_PRED].values.astype(float)
            mae, rmse, mp, n = calcular_metricas(y_real, y_pred, TRIGGER)
            if mae is not None:
                resultados[enlace] = {"MAE": mae, "RMSE": rmse, "MAPE": mp, "N": n}
        except Exception as e:
            print(f"   ❌ {f}: {e}")
    return resultados


def leer_baseline(etiqueta, carpeta_rel, prefijo, sufijo):
    col = f"{etiqueta}_t+1"
    carpeta_abs = os.path.join(ROOT_RESULTS, carpeta_rel)
    resultados = {}
    if not os.path.exists(carpeta_abs):
        return resultados
    for f in sorted(os.listdir(carpeta_abs)):
        if not f.startswith(prefijo) or not f.endswith(sufijo):
            continue
        enlace = f[len(prefijo): len(f) - len(sufijo)]
        if not enlace or not es_enlace_valido(enlace):
            continue
        try:
            df = pd.read_csv(os.path.join(carpeta_abs, f))
            if COL_REAL not in df.columns or col not in df.columns:
                continue
            y_real = df[COL_REAL].values.astype(float)
            y_pred = df[col].values.astype(float)
            mae, rmse, mp, n = calcular_metricas(y_real, y_pred, TRIGGER)
            if mae is not None:
                resultados[enlace] = {"MAE": mae, "RMSE": rmse, "MAPE": mp, "N": n}
        except Exception as e:
            print(f"   ❌ {f}: {e}")
    return resultados


def construir_tabla_enlaces(datos, cols_modelo, metricas=("MAE", "RMSE", "MAPE")):
    todos_enlaces = sorted(set(e for res in datos.values() for e in res))
    filas = []
    for enlace in todos_enlaces:
        fila = {"Enlace": enlace}
        for m in cols_modelo:
            if enlace in datos.get(m, {}):
                for met in metricas:
                    fila[f"{met}_{m}"] = datos[m][enlace][met]
            else:
                for met in metricas:
                    fila[f"{met}_{m}"] = None
        filas.append(fila)
    return pd.DataFrame(filas)


def ordenar_columnas_modelo(cols_modelo):
    baselines_orden = ["ARIMA", "Naive"]
    baselines = [c for c in baselines_orden if c in cols_modelo]
    arq_cols  = [c for c in cols_modelo if c not in baselines_orden]

    def sort_key(etiq):
        m = re.match(r"Arq(\d+)_([AB])", etiq)
        if m:
            return (int(m.group(1)), 0 if m.group(2) == "A" else 1)
        return (99, 99)

    return baselines + sorted(arq_cols, key=sort_key)


def imprimir_1v1(df, cols_modelo, ventana, filas_csv):
    baselines   = ["ARIMA", "Naive"]
    modelos_arq = [m for m in cols_modelo if m not in baselines]
    metricas    = ["MAE", "RMSE", "MAPE"]
    ancho       = 11

    for baseline in baselines:
        print(f"\n{'='*90}")
        print(f"  1 vs 1  ──  {baseline}  |  W={ventana}")
        print(f"{'='*90}")

        cabecera = f"{'Modelo':>16}"
        for met in metricas:
            cabecera += f"  {(met+' Gana'):>{ancho}}  {(met+' Pierde'):>{ancho}}  {(met+' %Win'):>{ancho}}"
        print(cabecera)
        print("-" * len(cabecera))

        for modelo in modelos_arq:
            line = f"{modelo:>16}"
            fila_csv = {"W": ventana, "Baseline": baseline, "Modelo": modelo}

            for met in metricas:
                col_mod  = f"{met}_{modelo}"
                col_base = f"{met}_{baseline}"
                if col_mod not in df.columns or col_base not in df.columns:
                    line += f"  {'?':>{ancho}}  {'?':>{ancho}}  {'?':>{ancho}}"
                    fila_csv[f"{met}_Gana"] = None
                    fila_csv[f"{met}_Pierde"] = None
                    fila_csv[f"{met}_Pct"] = None
                    continue

                sub    = df[[col_mod, col_base]].dropna()
                total  = len(sub)
                gana   = int((sub[col_mod] < sub[col_base]).sum())
                pierde = int((sub[col_mod] > sub[col_base]).sum())
                pct    = round(100 * gana / total, 1) if total > 0 else 0.0

                line += f"  {f'{gana}/{total}':>{ancho}}  {f'{pierde}/{total}':>{ancho}}  {f'{pct}%':>{ancho}}"
                fila_csv[f"{met}_Gana"]   = gana
                fila_csv[f"{met}_Pierde"] = pierde
                fila_csv[f"{met}_Total"]  = total
                fila_csv[f"{met}_Pct"]    = pct

            print(line)
            filas_csv.append(fila_csv)

        print(f"\n  G = modelo < {baseline}  |  P = modelo > {baseline}  |  % = victorias sobre {len(df)} enlaces")


def comparar():
    grupos = {}

    for nombre in sorted(os.listdir(ROOT_RESULTS)):
        m = PATRON_ARQ.match(nombre)
        if not m:
            continue
        arq_id, modo, w = m.group(1), m.group(2), m.group(3)
        ventana  = int(w)
        etiqueta = f"Arq{arq_id}_{modo}"

        if ventana not in grupos:
            grupos[ventana] = {}

        carpeta_abs     = os.path.join(ROOT_RESULTS, nombre)
        prefijo_fichero = f"test_Arq{arq_id}_1h_{modo}_W{w}_"
        print(f"📂 {nombre}  →  {etiqueta}")
        grupos[ventana][etiqueta] = leer_carpeta_arq(carpeta_abs, prefijo_fichero)
        print(f"   {len(grupos[ventana][etiqueta])} enlaces")

    if not grupos:
        print("❌ No se encontraron carpetas results_Arq*_1h_[AB]_W*")
        return

    filas_csv = []

    for ventana, datos_arq in sorted(grupos.items()):
        print(f"\n{'#'*60}")
        print(f"  W={ventana} | H=1h  —  {len(datos_arq)} modelos")
        print(f"{'#'*60}")

        datos_all = dict(datos_arq)
        for etiq, carp_rel, pref, suf in FUENTES_BASELINE:
            datos_all[etiq] = leer_baseline(etiq, carp_rel, pref, suf)

        cols_baselines = [e for e, *_ in FUENTES_BASELINE]
        cols_modelo    = ordenar_columnas_modelo(cols_baselines + list(datos_arq.keys()))
        df_tabla       = construir_tabla_enlaces(datos_all, cols_modelo)

        imprimir_1v1(df_tabla, cols_modelo, ventana, filas_csv)

    if filas_csv:
        df_out = pd.DataFrame(filas_csv)
        salida = f"{ROOT_RESULTS}/1v1_resumen_1h.csv"
        df_out.to_csv(salida, index=False)
        print(f"\n💾 CSV guardado: {salida}")


if __name__ == "__main__":
    comparar()

📂 results_Arq1_1h_A_W168  →  Arq1_A
   81 enlaces
📂 results_Arq1_1h_A_W24  →  Arq1_A
   81 enlaces
📂 results_Arq1_1h_B_W168  →  Arq1_B
   81 enlaces
📂 results_Arq1_1h_B_W24  →  Arq1_B
   81 enlaces
📂 results_Arq2_1h_A_W168  →  Arq2_A
   81 enlaces
📂 results_Arq2_1h_A_W24  →  Arq2_A
   81 enlaces
📂 results_Arq2_1h_B_W168  →  Arq2_B
   81 enlaces
📂 results_Arq2_1h_B_W24  →  Arq2_B
   81 enlaces
📂 results_Arq3_1h_A_W168  →  Arq3_A
   81 enlaces
📂 results_Arq3_1h_A_W24  →  Arq3_A
   81 enlaces
📂 results_Arq3_1h_B_W168  →  Arq3_B
   81 enlaces
📂 results_Arq3_1h_B_W24  →  Arq3_B
   81 enlaces
📂 results_Arq4_1h_A_W168  →  Arq4_A
   81 enlaces
📂 results_Arq4_1h_A_W24  →  Arq4_A
   81 enlaces
📂 results_Arq4_1h_B_W168  →  Arq4_B
   81 enlaces
📂 results_Arq4_1h_B_W24  →  Arq4_B
   81 enlaces

############################################################
  W=24 | H=1h  —  8 modelos
############################################################

  1 vs 1  ──  ARIMA  |  W=24
          Modelo     MAE Ga